# Evidently Data Drift Monitoring Dashboard

This notebook creates a data drift dashboard using Evidently AI to monitor:
- Feature drift between training and production data
- Target drift
- Model performance degradation
- Data quality issues

In [36]:
import boto3
import pandas as pd
from io import StringIO
import os
from dotenv import load_dotenv
import numpy as np
import evidently
from evidently import Report
from evidently.presets import DataDriftPreset, DataSummaryPreset

load_dotenv()

True

In [37]:
# Load data from S3
bucket_name = 'my-feature-store-data'
s3_key = 'pipeline-data/data.csv'

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
)

response = s3.get_object(Bucket=bucket_name, Key=s3_key)
csv_data = response['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(csv_data))

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

Dataset shape: (4902, 25)

Columns: ['index', 'aqi_index', 'co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'dew_point_2m', 'apparent_temperature', 'shortwave_radiation', 'et0_fao_evapotranspiration', 'year', 'month', 'day', 'hour', 'Calculated_AQI']


In [38]:
# Split data into reference (training) and current (test) sets
# Reference: First 80% (represents historical/training data)
# Current: Last 20% (represents new/production data)

split_index = int(len(df) * 0.8)
reference_data = df.iloc[:split_index].copy()
current_data = df.iloc[split_index:].copy()

print(f"Reference dataset: {reference_data.shape}")
print(f"Current dataset: {current_data.shape}")

Reference dataset: (3921, 25)
Current dataset: (981, 25)


In [39]:
# Define column mapping for Evidently (compatible with Evidently 0.7+)
# We'll choose the target column dynamically if both candidates exist

target_candidates = ['aqi_index', 'Calculated_AQI']
target_col = None
if 'df' in locals():
    for c in target_candidates:
        if c in df.columns:
            target_col = c
            break

column_mapping = {
    "target": target_col,               # Will be set to whichever exists in the dataframe (or None)
    "prediction": None,                 # Add predictions if you have them
    "numerical_features": [
        'co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3',
        'temperature_2m', 'relative_humidity_2m', 'precipitation',
        'wind_speed_10m', 'wind_direction_10m', 'surface_pressure',
        'dew_point_2m', 'apparent_temperature', 'shortwave_radiation',
        'et0_fao_evapotranspiration'
    ]
}

print("Column mapping configured successfully!")
print("Selected target:", target_col)

Column mapping configured successfully!
Selected target: aqi_index


In [40]:
# Create Data Drift Report
# Using Evidently 0.7+ presets: DataDriftPreset and DataSummaryPreset

# Optional: inspect Report.run signature for reference
import inspect
from evidently import Report as _ReportCls
print("Report.run signature:", inspect.signature(_ReportCls.run))

# Build report
data_drift_report = Report(metrics=[
    DataDriftPreset(),
    DataSummaryPreset(),
])

# Run report (Evidently 0.7+ returns a Snapshot)
report_snapshot = data_drift_report.run(
    reference_data=reference_data,
    current_data=current_data
)

print("Data drift report generated successfully!")

Report.run signature: (self, current_data: Union[ForwardRef('Dataset'), pandas.core.frame.DataFrame], reference_data: Union[ForwardRef('Dataset'), pandas.core.frame.DataFrame, NoneType] = None, additional_data: Optional[Dict[str, Union[ForwardRef('Dataset'), pandas.core.frame.DataFrame]]] = None, timestamp: Optional[datetime.datetime] = None, metadata: Dict[str, Union[str, Dict[str, str], List[str]]] = None, tags: List[str] = None, name: Optional[str] = None) -> evidently.core.report.Snapshot


c:\Users\Dell\Documents\Mlops-Project\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning:

invalid value encountered in divide

c:\Users\Dell\Documents\Mlops-Project\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning:

invalid value encountered in divide

c:\Users\Dell\Documents\Mlops-Project\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning:

invalid value encountered in divide

c:\Users\Dell\Documents\Mlops-Project\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning:

invalid value encountered in divide

c:\Users\Dell\Documents\Mlops-Project\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning:

invalid value encountered in divide

c:\Users\Dell\Documents\Mlops-Project\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning:

invalid value encountered in divide



Data drift report generated successfully!


In [41]:
# Save report as HTML (Evidently 0.7+ compatible)
import json
report_path = "../monitoring/evidently/reports/data_drift_report.html"
os.makedirs(os.path.dirname(report_path), exist_ok=True)

if hasattr(report_snapshot, "save_html"):
    report_snapshot.save_html(report_path)
    print(f"Report saved to: {report_path}")
elif hasattr(report_snapshot, "as_html"):
    html_str = report_snapshot.as_html()
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(html_str)
    print(f"Report saved to: {report_path}")
else:
    # Fallback: save JSON snapshot if HTML export isn't available
    json_path = report_path.replace(".html", ".json")
    if hasattr(report_snapshot, "json") and callable(report_snapshot.json):
        with open(json_path, "w", encoding="utf-8") as f:
            f.write(report_snapshot.json())
    elif hasattr(report_snapshot, "dict") and callable(report_snapshot.dict):
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(report_snapshot.dict(), f)
    else:
        # Last resort: serialize via repr
        with open(json_path, "w", encoding="utf-8") as f:
            f.write(repr(report_snapshot))
    print(f"HTML export not available in Evidently {getattr(evidently, '__version__', 'unknown')}.\nSaved JSON snapshot to: {json_path}")

Report saved to: ../monitoring/evidently/reports/data_drift_report.html


In [42]:
# Display report inline (if in Jupyter)
data_drift_report

In [43]:
# Get drift metrics from the snapshot (Evidently 0.7+)
import json

if 'report_snapshot' not in globals():
    raise RuntimeError("report_snapshot not found. Please run the report cell first.")

# Obtain a Python dict from the snapshot
if hasattr(report_snapshot, 'dict') and callable(report_snapshot.dict):
    drift_metrics = report_snapshot.dict()
elif hasattr(report_snapshot, 'json') and callable(report_snapshot.json):
    drift_metrics = json.loads(report_snapshot.json())
else:
    drift_metrics = {}

# Extract key metrics (robust to schema changes)
print("\n=== Data Drift Summary ===")
try:
    metrics_list = drift_metrics.get('metrics', [])
    # Try to find a data drift-related metric entry
    dd_candidates = []
    for m in metrics_list:
        txt = " ".join(str(m.get(k, '')) for k in ['metric', 'type', 'metric_id'])
        if 'DataDriftPreset' in txt or 'drift' in txt.lower():
            dd_candidates.append(m)
    dd_item = dd_candidates[0] if dd_candidates else (metrics_list[0] if metrics_list else None)
    result = (dd_item or {}).get('result', {})

    num_drifted = (
        result.get('number_of_drifted_columns')
        or result.get('drifted_features_count')
        or result.get('n_drifted_features')
    )
    dataset_drift = (
        result.get('dataset_drift')
        or result.get('dataset_drift_detected')
    )

    print(f"Number of drifted features: {num_drifted if num_drifted is not None else 'N/A'}")
    print(f"Dataset drift detected: {dataset_drift if dataset_drift is not None else 'N/A'}")
except Exception as e:
    print("Could not parse drift summary:", type(e).__name__, str(e))
    if isinstance(drift_metrics, dict):
        print("Top-level keys:", list(drift_metrics.keys()))
        if metrics_list:
            print("Sample metric entry keys:", list((metrics_list[0] or {}).keys()))


=== Data Drift Summary ===
Number of drifted features: N/A
Dataset drift detected: N/A


In [44]:
# Create Evidently Workspace for monitoring dashboard
# This will run a local server at localhost:7000

from evidently.ui.workspace import Workspace

# Create workspace
workspace_path = "../monitoring/evidently/workspace"
os.makedirs(workspace_path, exist_ok=True)

ws = Workspace.create(workspace_path)

# Create a project
project = ws.create_project("AQI Weather Monitoring")
project.description = "Data drift monitoring for AQI and Weather prediction model"

# Add run (snapshot) to project for UI (Evidently 0.7+)
ws.add_run(project.id, report_snapshot)

print(f"\nEvidently workspace created at: {workspace_path}")
print("\nTo view the dashboard, run:")
print(f"  evidently ui --workspace {workspace_path} --port 7000")


Evidently workspace created at: ../monitoring/evidently/workspace

To view the dashboard, run:
  evidently ui --workspace ../monitoring/evidently/workspace --port 7000


## Next Steps

### To view the Evidently Dashboard:

1. Open a terminal and run:
   ```bash
   evidently ui --workspace ./monitoring/evidently/workspace --port 7000
   ```

2. Open your browser and navigate to:
   ```
   http://localhost:7000
   ```

3. You should see:
   - Data drift metrics
   - Feature distribution comparisons
   - Target drift analysis
   - Data quality issues

### Automated Monitoring:

For production, schedule this notebook to run periodically:
- Daily: Compare yesterday's data with training data
- Weekly: Generate drift reports
- Alert when drift exceeds threshold

In [45]:
# Debug: inspect available Report export methods
print("Report type:", type(data_drift_report))
for name in ["save_html", "as_html", "save", "show", "json", "dict", "as_dict", "dump", "serialize"]:
    print(name, "->", hasattr(data_drift_report, name))

Report type: <class 'evidently.core.report.Report'>
save_html -> False
as_html -> False
save -> False
show -> False
json -> False
dict -> False
as_dict -> False
dump -> False
serialize -> False


In [46]:
# Debug: inspect report_snapshot export methods
print("Snapshot type:", type(report_snapshot))
for name in ["save_html", "as_html", "save", "show", "json", "dict", "as_dict", "dump", "serialize", "to_html"]:
    print(name, "->", hasattr(report_snapshot, name))

Snapshot type: <class 'evidently.core.report.Snapshot'>
save_html -> True
as_html -> False
save -> False
show -> False
json -> True
dict -> True
as_dict -> False
dump -> False
serialize -> False
to_html -> False


In [47]:
# Inspect Evidently Workspace API to find the right method to add a report/snapshot
import inspect
from pprint import pprint
from evidently.ui.workspace import Workspace
from evidently import __version__ as ev_version

print("Evidently version:", ev_version)
print("\nWorkspace class public attributes:")
pprint([n for n in dir(Workspace) if not n.startswith("_")])

workspace_path = "../monitoring/evidently/workspace"
ws = Workspace.create(workspace_path)
print("\nWorkspace instance public attributes:")
pprint([n for n in dir(ws) if not n.startswith("_")])

print("\nCallable methods on instance:")
pprint([n for n in dir(ws) if callable(getattr(ws, n)) and not n.startswith("_")])

Evidently version: 0.7.15

Workspace class public attributes:
['add_dataset',
 'add_project',
 'add_run',
 'create',
 'create_project',
 'delete_project',
 'delete_run',
 'get_dashboard',
 'get_project',
 'get_run',
 'list_projects',
 'list_runs',
 'save_dashboard',
 'search_project',
 'update_project']

Workspace instance public attributes:
['add_dataset',
 'add_project',
 'add_run',
 'create',
 'create_project',
 'delete_project',
 'delete_run',
 'get_dashboard',
 'get_project',
 'get_run',
 'list_projects',
 'list_runs',
 'path',
 'save_dashboard',
 'search_project',
 'state',
 'update_project']

Callable methods on instance:
['add_dataset',
 'add_project',
 'add_run',
 'create',
 'create_project',
 'delete_project',
 'delete_run',
 'get_dashboard',
 'get_project',
 'get_run',
 'list_projects',
 'list_runs',
 'save_dashboard',
 'search_project',
 'update_project']
